# SV2 tuần 8 — kết quả Device–KV260 đã lưu

Notebook chỉ đọc export Git; không SSH, build, chạy model hay đo lại. Kernel PC
có NumPy/Matplotlib. Mở JupyterLab dùng chung ở repo root, chọn kernel PC và
chạy từng ô bằng **Shift+Enter**. Các phiên raw đã đo và output notebook làm
việc giữ riêng; không ghi đè bằng notebook mẫu này.


In [ ]:
from pathlib import Path
import hashlib, json, csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from html import escape
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'edge').is_dir() and (p/'contracts').is_dir())
DATA = REPO/'results/week8/edge'
for line in (DATA/'sha256sum_export.txt').read_text().splitlines():
    expected, name = line.split(maxsplit=1)
    p = (DATA/name).resolve()
    assert DATA.resolve() in p.parents
    assert hashlib.sha256(p.read_bytes()).hexdigest() == expected, name
summary = json.loads((DATA/'summary.json').read_text())
review = json.loads((DATA/'sv1_review.json').read_text())
requests = list(csv.DictReader((DATA/'requests.csv').open()))
print('Nguồn:', DATA)
print('Checksum 5 file export: OK; receiver review ghi riêng, không sửa receipt cũ')
print('Scope:', summary['scope'])
print('Device G2:', summary['device_g2_acceptance'])
print('Registry:', summary['nonce_registry_joint_acceptance'])
print('Accuracy độc lập:', summary['independent_test_accuracy'])


In [ ]:
def table(rows):
    if not rows: return
    cols = list(rows[0])
    head = ''.join('<th>'+escape(str(c))+'</th>' for c in cols)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(r.get(c)))+'</td>' for c in cols)+'</tr>' for r in rows)
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))
table([
    {'Phạm vi':'G2 chính thức', 'Request':200, 'Logits bit-match offline':summary['g2_bit_matches_offline']},
    {'Phạm vi':'Bổ sung 1/2/4', 'Request':21, 'Logits bit-match offline':summary['supplement_bit_matches_offline']},
    {'Phạm vi':'Offline lặp', 'Request':100, 'Logits bit-match offline':summary['offline_repeat_bit_matches']},
])
table(summary['per_class_reference_agreement'])
print('87/100 là agreement trên 100 mẫu tích hợp có chọn lọc, không phải accuracy ECG độc lập.')


In [ ]:
classes = summary['per_class_reference_agreement']
x = np.arange(len(classes))
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(x-.18, [r['reference_count'] for r in classes], .36, label='Số mẫu theo nhãn reference')
ax.bar(x+.18, [r['class_matches'] for r in classes], .36, label='KV260 khớp nhãn reference')
ax.set_xticks(x, [r['class'] for r in classes]); ax.set_ylabel('Số mẫu')
ax.set_title('Agreement reference theo lớp — 100 mẫu tích hợp'); ax.legend()
fig.tight_layout(); plt.show()


## Timing toàn 212 request

Native tail là root CPU+DPU, chưa gồm BLE/Python/IPC/model load. Edge RX→TX
kết thúc ở BlueZ write submission, không phải ACK Device hoặc latency E2E.
Các mốc ms phía Edge dùng cùng đồng hồ monotonic; không đồng bộ với Device.


In [ ]:
table([{'Phép đo':key, 'Đơn vị':'ms', **summary[key]}
       for key in ('tail_ms', 'edge_rx_to_tx_submit_ms')])
g2 = [r for r in requests if r['trial']=='g2_212']
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([int(r['request_index']) for r in g2], [float(r['tail_ms']) for r in g2], label='Native root tail')
ax.plot([int(r['request_index']) for r in g2], [float(r['edge_rx_to_tx_submit_ms']) for r in g2], label='Edge RX→TX submit')
ax.set(xlabel='Request trong phiên 212', ylabel='ms', title='Timing đã lưu'); ax.legend()
fig.tight_layout(); plt.show()


In [ ]:
table([{'Tên':k, 'SHA256':v} for k,v in summary['service_sha256'].items()])
print('Nguồn link/Device:', review['source'])
print('Link:', review['link'])
print('Receiver import:', review['receiver_import'])
print('TC7:', review['registry']['joint_confirmation'])
print('Root receipt FLOAT32 output không tự xác nhận fix_point nội bộ DPU=3; xem báo cáo.')
print('Tuần9: Set/Read PHY mọi reconnect, TX/RX2M trước request; Week8 chưa tự động bước này.')
